En lugar de definir $M$ como una constante rígida ($M = 10 \cdot B$), se define como el tamaño esperado de un subconjunto candidato elástico $P_\alpha$, controlado por un factor de sobre-muestreo $\alpha \approx 10$.

Sean:
- $N_L$: Tamaño del conjunto de datos etiquetados ($\vert{}\mathcal{D}_{labeled}\vert{}$).
- $N_U$: Tamaño del conjunto de datos no etiquetados ($\vert{}\mathcal{D}_{unlabeled}\vert{}$).
- $B$: Presupuesto de anotación por iteración (budget), con $B = 0.05 \times N_L$.
- $\alpha$: Factor de sobre-muestreo de incertidumbre ($\alpha = 10$).
- $p$: Proporción del pool no etiquetado a seleccionar:
$$p = \min\left(1.0, \frac{\alpha \cdot B}{N_U}\right)$$
- $q$: Cuantil objetivo de incertidumbre: $q = 1 - p$.
- $\epsilon$: Error relativo tolerado en el cálculo del cuantil (ej. $\epsilon = 0.01$ o $1\%$).

from pyspark.sql import functions as sql_f

**1. Parámetros del experimento**
ALPHA = 10  # Factor de sobre-muestreo (oversampling ratio)
EPSILON = 0.01  # 1% de error relativo en el cuantil

**Asumiendo que conoces N_labeled y N_unlabeled (o calculas N_unlabeled)**
N_labeled = 100  # Ejemplo
N_unlabeled = unlabeled_predictions.count()

**2. Definición del Budget B y la proporción del pool**
B = int(0.05 * N_labeled)
p = min(1.0, (ALPHA * B) / N_unlabeled)
quantile_target = 1.0 - p

**3. Cálculo del umbral distribuido mediante approxQuantile (Greenwald-Khanna)**
threshold_val = unlabeled_predictions.stat.approxQuantile(
    "uncertainty", [quantile_target], EPSILON
)[0]

**4. Filtrado distribuido directo (M_approx ≈ ALPHA * B)**
uncertainty_candidates = unlabeled_predictions.filter(
    sql_f.col("uncertainty") >= threshold_val
)

Si necesitas justificar esta decisión en un entorno académico o informe técnico, puedes estructurar el argumento en tres pilares:
Las métricas de incertidumbre en aprendizaje automático (entropía, margen de clasificación, varianza en Monte Carlo Dropout) son estimaciones continuas con ruido. No existe un límite semántico real que diferencie el elemento ordenado en la posición $M$ del elemento $M+1$. Exigir un corte exacto en $M$ impone una frontera rígida artificial sobre una distribución probabilística continua.

B. Escalabilidad y Complejidad AlgorítmicaCorte Exacto ($top-M$ global):
- Requiere un ordenamiento global $O(N_U \log N_U)$ o un paso de sincronización hacia un único nodo driver, generando un cuello de botella de red (shuffle bottleneck) al desplazar vectores de características (features).
- Corte Aproximado (approxQuantile): Utiliza algoritmos de resúmenes de flujo distribuidos (como Greenwald-Khanna). Su complejidad es de $O(N_U \log(1/\epsilon))$, ejecuta de forma $100\%$ paralela en los executors y reduce la transferencia en red a un único valor escalar (threshold_val).

C. Robustez de la Fase 2 (Selección por Diversidad)La Fase 2 (diversidad) actúa sobre la geometría del espacio de características del conjunto reducido $P_\alpha$.

Un margen de fluctuación de $\pm (\epsilon \cdot N_U)$ elementos en el pool candidato no altera la distribución espacial de los candidatos ni degrada la selección de las $B$ muestras finales, pero reduce drásticamente el tiempo de computación por iteración de Active Learning.

In [2]:
def get_partition_farthest(partition):
  """Cada worker devuelve únicamente la muestra con mayor min_distance de su partición."""
  farthest = None
  for row in partition:
    # row[0]: id_sample, row[1]: features, row[2]: min_distance
    if farthest is None or row[2] > farthest[2]:
      farthest = row

  if farthest is not None:
    yield farthest

In [3]:
from pyspark.ml.linalg import Vectors


def update_min_distance(
    partition, new_selected_id, new_selected_features
):
  """Calcula la distancia de 18 floats directamente usando Vectors.squared_distance."""
  for row in partition:
    # Descartar la muestra recién elegida
    if row[0] == new_selected_id:
      continue

    # row[1] es DenseVector de tamaño 18, new_selected_features es DenseVector de tamaño 18
    dist = Vectors.squared_distance(row[1], new_selected_features)

    # Actualización del mínimo
    yield (row[0], row[1], min(row[2], dist))

In [ ]:
def farthest_first_selection(candidates_df, batch_size, sc):
    """Selección Farthest-First distribuida y optimizada.

    candidates_df: DataFrame con [id_sample, features, uncertainty] batch_size:
    Número de muestras a seleccionar (B) sc: SparkContext activo
    """
    # ---------------------------------------------------------
    # 1. Primer seleccionado: Mayor incertidumbre
    # ---------------------------------------------------------
    first_candidate = candidates_df.orderBy(sql_f.col("uncertainty").desc()).first()
    first_id = first_candidate["id_sample"]
    first_features = first_candidate["features"]

    selected = [first_id]

    # ---------------------------------------------------------
    # 2. Inicializar RDD de candidatos restantes
    # ---------------------------------------------------------
    candidates_df_filter = candidates_df.filter(
        sql_f.col("id_sample") != first_id
    )

    # Creación del RDD inicial: (id_sample, features, min_distance)
    candidates_rdd = candidates_df_filter.rdd.map(
        lambda row: (
            row["id_sample"],
            row["features"],
            Vectors.squared_distance(row["features"], first_features),
        )
    )
    # Uso de local checkpoint
    candidates_rdd.localCheckpoint()

    # ---------------------------------------------------------
    # 3. Bucle Iterativo Farthest-First
    # ---------------------------------------------------------
    # Mantenemos una referencia al broadcast activo
    active_broadcast = None
    for iteration in range(1, batch_size):

        # Worker: Obtiene el más lejano local de cada partición
        partition_candidates = candidates_rdd.mapPartitions(get_partition_farthest)

        # Driver: Obtiene el máximo global entre las respuestas de las particiones
        # CORREGIDO: row[2] contiene min_distance
        farthest_candidate = partition_candidates.max(key=lambda row: row[2])
        # 2. AHORA SÍ ES SEGURO: Una vez que .collect() terminó, destruimos
        #    el broadcast que se utilizó para calcular esa iteración
        if active_broadcast is not None:
            active_broadcast.destroy()

        farthest_id = farthest_candidate[0]
        farthest_features = farthest_candidate[1]

        # Se añade el id a selected
        selected.append(farthest_id)

        # Broadcast de las features del nuevo seleccionado
        active_broadcast = sc.broadcast(farthest_features)

        # Se actualizan las distancias respecto al nuevo seleccionado
        candidates_rdd = candidates_rdd.mapPartitions(
            lambda partition:
                update_min_distance(
                    partition,
                    farthest_id,
                    active_broadcast.value
                )
        )

        candidates_rdd.localCheckpoint()

        print(
            f"Iteración {iteration + 1}/{batch_size} - Seleccionado id:"
            f" {farthest_id}"
        )

    return selected

1. ¿Qué hace exactamente localCheckpoint()?
Realiza dos acciones principales:
- Guarda el contenido del RDD en el almacenamiento local (memoria/disco) de los propios executors que están procesando cada partición.
- Trunca (corta) el grafo de linaje (DAG): Le dice a Spark que "olvide" todo el historial de transformaciones previas que se usaron para calcular ese RDD y que considere este nuevo RDD como la fuente de datos original.

2. ¿Para qué sirve en tu implementación?
En Spark, debido a la evaluación perezosa (lazy evaluation), Spark no guarda los datos transformados por defecto; en su lugar, guarda la receta/instrucciones de cómo calcularlos. Esa receta es el DAG (Directed Acyclic Graph) o linaje.

El problema sin localCheckpoint():
Si tienes un bucle de $B = 100$ iteraciones donde en cada paso haces:
candidates_rdd = candidates_rdd.mapPartitions(...)
Spark va encadenando las transformaciones:
$$\text{RDD}_0 \to \text{RDD}_1 \to \text{RDD}_2 \to \dots \to \text{RDD}_{100}$$
En la iteración 100, la "receta" tiene 100 niveles de profundidad. Cuando Spark intenta evaluar esa tarea:

El Driver gasta muchísima memoria serializando un grafo gigante.
- La Máquina Virtual de Java (JVM) del Driver suele lanzar un StackOverflowError porque la cadena de llamadas de la receta es demasiado larga.
La solución con localCheckpoint():
-Al ejecutar localCheckpoint() al final de cada iteración, el linaje se reinicia:
- Iteración 1: Se calcula y se corta linaje $\to$ El DAG vuelve a tener profundidad $0$.
- Iteración 2: Se calcula y se corta linaje $\to$ El DAG vuelve a tener profundidad $0$.
Gracias a esto, el programa puede ejecutar 10.000 iteraciones sin que la memoria del Driver sufra ni colapse.